# torch-compress


PyTorch port of [jax-compress](https://github.com/byronknoll/jax-compress) by Byron Knoll. Same compression algorithm, neural network, and arithmetic coder as the JAX/Flax version; the model, training loop, and checkpoint format have been rewritten with `torch.nn`.


## Parameters

In [ ]:
#@title Parameters
#@markdown ---
batch_size = 128 #@param {type:"integer"}
#@markdown _Splits the file into N batches to process them in parallel. Increasing this value improves speed but may negatively affect the compression rate. For optimal speed on certain GPUs, set this to a multiple of 8._

#@markdown ---
seq_length =  15 #@param {type:"integer"}
#@markdown _Determines the horizon for backpropagation through time. Lowering this value improves speed but can reduce the compression rate._

#@markdown ---
rnn_units =  1400 #@param {type:"integer"}
#@markdown _The number of units to use within each LSTM layer. Lowering this improves speed and reduces memory usage, but can worsen the compression rate. Make this a multiple of 8 to improve speed on certain GPUs._

#@markdown ---
num_layers = 8 #@param {type:"integer"}
#@markdown _The number of LSTM layers to use. Lowering this value improves speed at the cost of the compression rate._

#@markdown ---
embedding_size=512 #@param {type:"integer"}
#@markdown _The size of the embedding layer._

#@markdown ---
ensemble_size = 1 #@param {type:"integer"}
#@markdown _The number of LSTM models to use in the ensemble._

#@markdown ---
learning_rate_schedule = "0:0.0005 200000:0.0002" #@param {type:"string"}
#@markdown _The learning rate schedule formatted as "step:value". Values are linearly interpolated between points. Special handling is applied for the first and last points: from the start of the file to the first point, the value remains constant. Similarly, it stays constant from the last data point to the end of the file._

#@markdown ---
adam_b1 = 0.0 #@param {type:"number"}
#@markdown _The Beta1 parameter for the Adam optimizer._

#@markdown ---
adam_b2 = 0.9999 #@param {type:"number"}
#@markdown _The Beta2 parameter for the Adam optimizer._

#@markdown ---
adam_eps = 1e-12 #@param {type:"number"}
#@markdown _The Epsilon parameter for the Adam optimizer._

#@markdown ---
mode = 'compress' #@param ["compress", "decompress", "both", "preprocess_only"]
#@markdown _Select the execution mode. "preprocess_only" will exclusively run preprocessing and skip the compression step._

#@markdown ---
preprocess = 'nncp' #@param ["cmix", "nncp", "nncp-done", "none"]
#@markdown _Select the preprocessor. NNCP performs better on enwik8/enwik9. It is slower because it builds a custom dictionary, whereas cmix uses a pretrained one. "nncp-done" is for files already preprocessed by NNCP (the dictionary must be included)._

#@markdown ---
n_words = 8192 #@param {type:"integer"}
#@markdown _For NNCP preprocessor only: the approximate maximum dictionary size. Recommended values: 8192 for enwik8 and enwik9._

#@markdown ---
min_freq = 64 #@param {type:"integer"}
#@markdown _For NNCP preprocessor only: the minimum frequency required for selected words. Recommended values: 64 for enwik8, 512 for enwik9._

#@markdown ---
path_to_file = "enwik8" #@param ["enwik4", "enwik6", "enwik8", "enwik9", "custom"]
#@markdown _The name of the file to compress or decompress. If "custom" is selected, use the next parameter to specify your custom file path._

#@markdown ---
custom_path = '' #@param {type:"string"}
#@markdown _Used when "path_to_file" is set to "custom". Specify the name of the file you want to compress or decompress. You can transfer files using the "http_path" or "local_upload" options below._

#@markdown ---
http_path = '' #@param {type:"string"}
#@markdown _Selects the URL to download your file from. Using Google Drive URLs is recommended for faster transfer speeds. Use the format "https://drive.google.com/uc?id=<FILE_ID>". You can find the file ID from the "Get Link" URL in Google Drive. You can enter multiple URLs separated by spaces._

#@markdown ---
local_upload = False #@param {type:"boolean"}
#@markdown _If enabled, prompts you in the "Setup Files" section to upload files directly from your computer. Note: Uploading locally can be slow compared to using "http_path"._

#@markdown ---
download_option = "no_download" #@param ["no_download", "local", "google_drive"]
#@markdown _Select how to handle output files. "local" downloads directly to your device. "google_drive" copies them to your Google Drive for much faster transfer speeds._

#@markdown ---
checkpoint = False #@param {type:"boolean"}
#@markdown _If enabled, a checkpoint of model weights will be saved according to your "download_option". This is useful for avoiding Colab session timeouts by splitting files into segments and saving/loading model weights between runs. Existing checkpoints load automatically at the start of compression._

#@markdown ---
total_parts = 1 #@param {type:"integer"}
#@markdown _The total number of segments to split file processing into across several sessions._

#@markdown ---
current_part = 1 #@param {type:"integer"}
#@markdown _The current file part to process (range: 1 to total_parts)._

#@markdown ---
retrain_period_schedule = "0:1001 200000:5001" #@param {type:"string"}
#@markdown _The retrain period schedule formatted as "step:value". Values are linearly interpolated between points._

#@markdown ---
retrain_block_len = 100000 #@param {type:"integer"}
#@markdown _Retrain over the last M symbols._

#@markdown ---
retrain_seq_length = 100 #@param {type:"integer"}
#@markdown _The sequence length used during retraining. This permits a longer horizon during retraining steps compared to online compression._

#@markdown ---
retrain_batch_size = 256 #@param {type:"integer"}
#@markdown _The batch size designated for retraining. Increasing this improves parallelism during the retraining phase._

#@markdown ---
retrain_lr_schedule = "0:0.0005 200000:0.0002" #@param {type:"string"}
#@markdown _The retraining learning rate schedule formatted as "step:value". Values are linearly interpolated between points._

#@markdown ---
retrain_dropout = 0.4 #@param {type:"number"}
#@markdown _The dropout rate applied during retraining._
#@markdown ---
tensorboard = False #@param {type:"boolean"}
#@markdown _If enabled, logs training metrics (bpc, loss, lr, retrain loss) to TensorBoard during compression / decompression._

#@markdown ---
tensorboard_run_name = "torch_compress" #@param {type:"string"}
#@markdown _Subdirectory name for this run inside `tensorboard_logdir`. Use distinct names to compare runs in TensorBoard._

#@markdown ---
tensorboard_logdir = "data/tensorboard" #@param {type:"string"}
#@markdown _Parent directory for TensorBoard event files. Run `tensorboard --logdir <logdir>` to view all runs side by side._

#@markdown ---
use_bf16 = True #@param {type:"boolean"}
#@markdown _Run the model forward/backward in bfloat16 mixed precision. ~2x faster on bf16-capable GPUs (Ampere/Hopper, GH200/H100). Compress and decompress MUST use the same setting -- a file compressed with use_bf16=True can only be decompressed with use_bf16=True (and vice versa)._

#@markdown ---
model_type = "lstm" #@param ["lstm", "transformer_xl"]
#@markdown _Predictor architecture. "lstm" is the default reference model. "transformer_xl" routes through the NNCP-style adapter (`models.transformer_xl.TransformerXLModel`) and requires the local repo (the `models/` package must be importable -- not available in a vanilla Colab clone-less session)._

#@markdown ---
#@markdown ### Transformer-XL hparams
#@markdown _Only consulted when `model_type == "transformer_xl"`. Defaults track NNCP v2's `nncp_enwik_base.sh` config._
n_layer = 12 #@param {type:"integer"}
n_head = 8 #@param {type:"integer"}
d_model = 512 #@param {type:"integer"}
d_head = 64 #@param {type:"integer"}
d_inner = 2048 #@param {type:"integer"}
mem_len = 160 #@param {type:"integer"}
ext_tgt_len = 31 #@param {type:"integer"}
attn_type = 1 #@param {type:"integer"}
tied_r_bias = True #@param {type:"boolean"}
use_gelu = True #@param {type:"boolean"}
dropout = 0.25 #@param {type:"number"}
dropatt = 0.0 #@param {type:"number"}
init_std = 0.013 #@param {type:"number"}
retrain_tgt_len = 64 #@param {type:"integer"}
retrain_mem_len = 128 #@param {type:"integer"}
#@markdown _The model is constructed with `dropout`, but streaming forward passes use `deterministic=True` (eval mode, dropout off). Dropout activates only during retraining (`deterministic=False`, train mode). `retrain_tgt_len` and `retrain_mem_len` are NNCP's retrain-time shape — `model.reset_length()` swaps these in around each retrain pass and restores the streaming shape (1, 0, mem_len) afterward._

#@markdown ---
#@markdown _Transformer-XL learning-rate schedules. NNCP's published schedule (`nncp_enwik_base.sh`) was tuned for enwik9 at batch_size=64 -> ~1.5M streaming steps, with the first decay at 341K (~23% into the run). Our enwik8 runs at batch_size=128 do only ~202K steps, so NNCP's 341K transition would never fire. The default below pulls the transition to step 50K (~25% of a 200K-step run) and the second to 150K (~75%) so the LR actually decays during enwik8._
learning_rate_schedule_xl = "0:7.9e-5 50000:1.6e-5 150000:5.0e-6" #@param {type:"string"}
retrain_lr_schedule_xl = "0:4.0e-4 13000:2.0e-4 93000:1.0e-4 163000:5.0e-5 1911300:1.6e-5" #@param {type:"string"}

#@markdown ---
#@markdown ### Transformer-XL optimizer hparams (NNCP-aligned)
#@markdown _Used by `_process_transformer_xl` and the transformer half of `_process_hybrid`. NNCP-base uses `clip=0.25` (vs torch_compress's tuned 4.0 for the LSTM) and `adam_eps=1e-9` (vs torch_compress's 1e-12). Closer alignment with NNCP's published config._
clip_xl = 0.25 #@param {type:"number"}
adam_eps_xl = 1e-9 #@param {type:"number"}


## Setup

In [ ]:
#@title Imports

# --- LTCB determinism preamble ---------------------------------------------
# CUBLAS_WORKSPACE_CONFIG must be set before any CUDA initialization, otherwise
# torch.use_deterministic_algorithms will refuse to enable strict mode.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import torch
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# Force bf16 matmul accumulation in fp32 (default is bf16 reduction). Without
# this, parallel reductions in bf16 matmul kernels on CUDA produce bit-
# different outputs across kernel launches given identical inputs, which
# breaks the encode/decode probability agreement the arithmetic coder needs.
# No-op on fp32 runs; ~10-20% slowdown vs bf16-reduce on bf16 runs in
# exchange for round-trip-safe determinism.
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False
# ---------------------------------------------------------------------------

import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
try:
  from google.colab import files
except ImportError:
  files = None
import time
import math
import sys
import subprocess
import contextlib
from typing import Any, List, Tuple
try:
  from google.colab import drive
except ImportError:
  drive = None
import pickle


In [ ]:
#@title System Info

def system_info():
  """Prints out system information."""
  gpu_info = !nvidia-smi
  gpu_info = '\n'.join(gpu_info)
  if gpu_info.find('failed') >= 0:
    print('Select the Runtime → "Change runtime type" menu to enable a GPU accelerator, ')
    print('and then re-execute this cell.')
  else:
    print(gpu_info)
  print("PyTorch version:", torch.__version__)
  print("CUDA available:", torch.cuda.is_available())
  if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
  !lscpu |grep 'Model name'
  !cat /proc/meminfo | head -n 3

system_info()


In [ ]:
#@title Mount Google Drive
if download_option == "google_drive":
  drive.mount('/content/gdrive')

In [ ]:
#@title Setup Files

!mkdir -p "data"

if local_upload:
  %cd data
  files.upload()
  %cd ..

if path_to_file == 'enwik8' or path_to_file == 'enwik6' or path_to_file == 'enwik4':
  %cd data
  # Download enwik8 dataset
  !gdown --id 11-twesB-vGexGZpVSFbaCZDXKh5jMmMd
  # Create smaller subsets of the dataset
  !head -c 1000000 enwik8 > enwik6
  !head -c 10000 enwik8 > enwik4
  path_to_file = 'data/' + path_to_file
  %cd ..

if path_to_file == 'enwik9':
  %cd data
  # Download and extract the full enwik9 dataset
  !gdown --id 1D2gCmf9AlXIBP62ARhy0XcIuIolOTRAE
  !unzip enwik9.zip
  path_to_file = 'data/' + path_to_file
  %cd ..

if path_to_file == 'custom':
  path_to_file = 'data/' + custom_path

if http_path:
  %cd data
  paths = http_path.split()
  # Download any custom HTTP paths provided
  for path in paths:
    !gdown $path
  %cd ..

if preprocess == 'cmix':
  # Download and extract the cmix preprocessor
  !gdown --id 1qa7K28tlUDs9GGYbaL_iE9M4m0L1bYm9
  !unzip cmix-v18.zip
  %cd cmix
  !make
  %cd ..

if preprocess == 'nncp' or preprocess == 'nncp-done':
  # Download and extract the NNCP preprocessor
  !gdown --id 1VzFEzfN3Id0F32JygxelMPyOxGpIabL4
  !unzip nncp.zip
  %cd nncp/
  !make preprocess
  %cd ..

if checkpoint:
    ckpt_path = "data/ckpt.zip"
    if os.path.exists(ckpt_path):
        print(f"Unzipping model checkpoint from {ckpt_path}...")
        !unzip -o {ckpt_path} -d data/ckpt
    else:
        print(f"Checkpoint flag is enabled but {ckpt_path} was not found. Please upload it if you are resuming a session.")

if current_part > 1:
    partial_compressed_path = "data/compressed.dat"
    if not os.path.exists(partial_compressed_path):
        print(f"Warning: Attempting to resume compression (part {current_part}), but {partial_compressed_path} could not be found.")
        print("Please ensure you have uploaded the partially compressed file from your previous session.")
    else:
        print(f"Successfully found the partial compressed file: {partial_compressed_path}")

In [ ]:
#@title Model Architecture

class LSTMModel(nn.Module):
  """LSTM stack with skip connections from the embedding into every layer's input.

  Mirrors the Flax model: each layer (after the first) sees concat(embedding, prev_output);
  the dense head sees concat of all layers' outputs.

  Implementation note: uses nn.LSTMCell in a manual time loop instead of nn.LSTM.
  cuDNN's RNN kernel is non-deterministic even with cudnn.deterministic=True, which
  would break the LTCB compress/decompress invariant. The cell-loop is slower but
  produces bit-identical predictions across runs on the same machine.
  """

  def __init__(self, vocab_size, embedding_size, rnn_units, num_layers, dropout_rate=0.0):
    super().__init__()
    self.vocab_size = vocab_size
    self.embedding_size = embedding_size
    self.rnn_units = rnn_units
    self.num_layers = num_layers
    self.dropout_rate = dropout_rate

    self.embed = nn.Embedding(vocab_size, embedding_size)

    self.lstm_cells = nn.ModuleList()
    for i in range(num_layers):
      input_size = embedding_size if i == 0 else embedding_size + rnn_units
      self.lstm_cells.append(nn.LSTMCell(input_size, rnn_units))

    out_size = rnn_units * num_layers if num_layers > 1 else rnn_units
    self.dense_logits = nn.Linear(out_size, vocab_size)

  def init_states(self, batch_size, device):
    """Returns a list of (h, c) tuples (one per layer), each (batch, rnn_units)."""
    return [
        (torch.zeros(batch_size, self.rnn_units, device=device),
         torch.zeros(batch_size, self.rnn_units, device=device))
        for _ in range(self.num_layers)
    ]

  def forward(self, inputs, states, return_sequence=False, deterministic=True):
    """
    Args:
      inputs: (batch, seq_length) int64
      states: list of (h, c) tuples per layer; each (batch, rnn_units)
      return_sequence: if True, returns logits for the entire sequence
      deterministic: if True, dropout is disabled (matches Flax's deterministic=True)
    Returns:
      logits: (batch, vocab) if not return_sequence else (batch, seq, vocab)
      new_states: list of (h, c) tuples per layer
    """
    use_dropout = not deterministic
    embedding = F.dropout(self.embed(inputs), p=self.dropout_rate, training=use_dropout)
    seq_len = embedding.shape[1]

    new_states = list(states)
    layer_outputs = []
    curr_input = embedding

    for i, cell in enumerate(self.lstm_cells):
      h, c = states[i]
      step_outputs = []
      for t in range(seq_len):
        h, c = cell(curr_input[:, t, :], (h, c))
        step_outputs.append(h)
      layer_out = torch.stack(step_outputs, dim=1)  # (batch, seq, rnn_units)
      new_states[i] = (h, c)

      if i < self.num_layers - 1:
        layer_out = F.dropout(layer_out, p=self.dropout_rate, training=use_dropout)
      layer_outputs.append(layer_out)

      if i < self.num_layers - 1:
        curr_input = torch.cat([embedding, layer_out], dim=-1)

    if return_sequence:
      final_rep = layer_outputs[0] if self.num_layers == 1 else torch.cat(layer_outputs, dim=-1)
      logits = self.dense_logits(final_rep)
    else:
      if self.num_layers == 1:
        final_rep = layer_outputs[0][:, -1, :]
      else:
        final_rep = torch.cat([o[:, -1, :] for o in layer_outputs], dim=-1)
      logits = self.dense_logits(final_rep)

    return logits.float(), new_states


def build_model(model_type, *, vocab_size, embedding_size, rnn_units, num_layers,
                dropout_rate=0.0):
  """Construct the predictor selected by ``model_type``.

  The default "lstm" returns a ``LSTMModel`` constructed identically to before,
  preserving the existing reference behaviour. "transformer_xl" lazy-imports
  the local NNCP-style adapter; it is unavailable in a vanilla Colab session
  that has not cloned this repo.

  Transformer hparams are read from notebook globals (``n_layer``, ``n_head``,
  ``d_model``, ``d_head``, ``d_inner``, ``mem_len``, ``ext_tgt_len``,
  ``attn_type``, ``tied_r_bias``, ``use_gelu``, ``dropout``, ``dropatt``,
  ``init_std``) so the LSTM call sites do not need to change. Missing globals
  fall back to the adapter's NNCP-base defaults.
  """
  if model_type == "lstm":
    return LSTMModel(vocab_size=vocab_size, embedding_size=embedding_size,
                     rnn_units=rnn_units, num_layers=num_layers,
                     dropout_rate=dropout_rate)
  if model_type == "transformer_xl":
    from models.transformer_xl import TransformerXLModel
    g = globals()
    return TransformerXLModel(
        vocab_size=vocab_size,
        n_layer=g.get('n_layer', 12),
        n_head=g.get('n_head', 8),
        d_model=g.get('d_model', 512),
        d_head=g.get('d_head', 64),
        d_inner=g.get('d_inner', 2048),
        dropout=g.get('dropout', 0.0),
        dropatt=g.get('dropatt', 0.0),
        tgt_len=1,  # streaming inference shape; reset_length toggles for retrain
        ext_len=g.get('ext_tgt_len', 31),
        mem_len=g.get('mem_len', 160),
        attn_type=g.get('attn_type', 1),
        tied_r_bias=g.get('tied_r_bias', True),
        use_gelu=g.get('use_gelu', True),
        init_std=g.get('init_std', 0.02),
    )
  if model_type == "hybrid":
    # Construct LSTM + Transformer-XL via the same code paths used for the
    # solo cases, then wrap them in HybridModel. Both submodels are unmodified;
    # the hybrid is purely composition.
    lstm = LSTMModel(vocab_size=vocab_size, embedding_size=embedding_size,
                     rnn_units=rnn_units, num_layers=num_layers,
                     dropout_rate=dropout_rate)
    from models.transformer_xl import TransformerXLModel
    g = globals()
    xl = TransformerXLModel(
        vocab_size=vocab_size,
        n_layer=g.get('n_layer', 12),
        n_head=g.get('n_head', 8),
        d_model=g.get('d_model', 512),
        d_head=g.get('d_head', 64),
        d_inner=g.get('d_inner', 2048),
        dropout=g.get('dropout', 0.0),
        dropatt=g.get('dropatt', 0.0),
        tgt_len=1,
        ext_len=g.get('ext_tgt_len', 31),
        mem_len=g.get('mem_len', 160),
        attn_type=g.get('attn_type', 1),
        tied_r_bias=g.get('tied_r_bias', True),
        use_gelu=g.get('use_gelu', True),
        init_std=g.get('init_std', 0.02),
    )
    from models.hybrid import HybridModel
    # Optional learned mixer (cmix-style context-dependent gating). When the
    # `use_learned_mixer` notebook global is True, attach a tiny MLP that
    # produces per-step weights from per-submodel confidence features
    # (entropy + max log-prob). When False, HybridModel.mixer is None and
    # the ensemble loop falls back to equal-weight geometric mean.
    mixer = None
    if g.get('use_learned_mixer', False):
      from models.learned_mixer import LearnedMixer
      mixer = LearnedMixer(
          n_models=2,
          hidden_dim=g.get('mixer_hidden_dim', 64),
          init_std=g.get('mixer_init_std', 0.02),
      )
    return HybridModel(lstm, xl, mixer=mixer)
  raise ValueError(f"unknown model_type: {model_type!r}")


In [ ]:
#@title Compression Library


def parse_schedule(schedule_str):
  """Parses a learning rate or training schedule string.

  Expects a string formatted as "step:value step:value" and
  returns a list of parsed (step, value) tuples sorted by step.
  """
  points = []
  try:
    for item in schedule_str.split():
      step, value = item.split(':')
      points.append((float(step), float(value)))
    points.sort(key=lambda x: x[0])
  except ValueError:
    print(f"Error parsing schedule: {schedule_str}")
    return []
  return points


def get_scheduled_value(schedule, step):
  """Linearly interpolates a value at the given step using the schedule.

  Outside the bounded range, the closest endpoint value is returned.
  """
  if not schedule:
    return 0.0
  if step <= schedule[0][0]:
    return schedule[0][1]
  if step >= schedule[-1][0]:
    return schedule[-1][1]
  for i in range(len(schedule) - 1):
    start_step, start_val = schedule[i]
    end_step, end_val = schedule[i+1]
    if start_step <= step <= end_step:
      fraction = (step - start_step) / (end_step - start_step)
      return start_val + fraction * (end_val - start_val)
  return schedule[-1][1]


def get_symbol(index, length, freq, coder, compress, data):
  """Reads or writes a single symbol via the arithmetic coder."""
  symbol = 0
  if index < length:
    if compress:
      symbol = data[index]
      coder.write(freq, symbol)
    else:
      symbol = coder.read(freq)
      data[index] = symbol
  return symbol


def reset_seed():
  SEED = 1234
  os.environ['PYTHONHASHSEED'] = str(SEED)
  random.seed(SEED)
  np.random.seed(SEED)
  torch.manual_seed(SEED)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def download(path):
  if download_option == 'local':
    files.download(path)
  elif download_option == 'google_drive':
    !cp -f $path /content/gdrive/My\ Drive


def detach_states(states_per_model):
  """Detach a list (over ensemble) of lists (over layers) of (h, c) tuples."""
  return [
      [(h.detach(), c.detach()) for (h, c) in layer_states]
      for layer_states in states_per_model
  ]


def _lstm_autocast(device):
  """Returns a bf16 autocast context if ``use_bf16`` is set and CUDA is available.

  When disabled, returns a no-op autocast context so call sites stay uniform.
  Parameters and the loss/backward stay in fp32; only the forward matmuls inside
  ``LSTMModel`` run in bf16. The model's ``return logits.float()`` upcasts the
  logits at the boundary so the arithmetic coder always sees fp32 probabilities.
  """
  use_bf16_flag = bool(globals().get('use_bf16', False)) and torch.cuda.is_available()
  ac_dtype = torch.bfloat16 if use_bf16_flag else torch.float32
  return torch.autocast(device_type=device.type, dtype=ac_dtype, enabled=use_bf16_flag)


def forward_ensemble(models, inputs, states_per_model):
  """Runs forward pass for each ensemble member with autograd enabled.

  Returns:
    probs_np: numpy array (batch, vocab), geometric-mean ensemble probabilities
    logits_list: list of (batch, vocab) tensors with grad
    new_states_per_model: list (over ensemble) of lists of (h, c)
  """
  log_probs_list = []
  logits_list = []
  new_states_per_model = []
  for model, states in zip(models, states_per_model):
    # deterministic=True matches the JAX path used for online updates (no dropout).
    # We deliberately do NOT call model.eval(): cuDNN LSTM requires train() mode for backward.
    with _lstm_autocast(inputs.device):
      logits, new_states = model(inputs, states, return_sequence=False, deterministic=True)
    logits_list.append(logits)
    log_probs_list.append(F.log_softmax(logits, dim=-1))
    new_states_per_model.append(new_states)
  mean_log_probs = torch.stack(log_probs_list, dim=0).mean(dim=0)
  probs = F.softmax(mean_log_probs, dim=-1).detach().cpu().numpy()
  return probs, logits_list, new_states_per_model


def backward_and_step(logits_list, optimizers, models, symbols, mask, vocab_size,
                      clip=4.0):
  """Cross-entropy backward + gradient clip + optimizer step for every ensemble member.

  Returns the geometric-mean ensemble loss (scalar) and the mask sum (denominator).

  ``clip`` defaults to 4.0 (torch_compress's tuned value for the LSTM); the
  Transformer-XL streaming path passes clip=clip_xl (NNCP-aligned 0.25).
  """
  log_probs_list = []
  for logits, optimizer, model in zip(logits_list, optimizers, models):
    optimizer.zero_grad()
    loss_per = F.cross_entropy(logits, symbols, reduction='none')
    loss = (loss_per * mask).sum()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
    optimizer.step()
    log_probs_list.append(F.log_softmax(logits.detach(), dim=-1))

  mean_log_probs = torch.stack(log_probs_list, dim=0).mean(dim=0)
  one_hot = F.one_hot(symbols, num_classes=vocab_size).float()
  loss_vals = -(one_hot * F.log_softmax(mean_log_probs, dim=-1)).sum(dim=-1)
  return (loss_vals * mask).sum().item(), mask.sum().item()


def retrain_step(models, retrain_optimizers, inputs, targets, current_lr):
  """Runs a single retraining step (forward + backward) over a full sequence with dropout."""
  losses = []
  for model, optimizer in zip(models, retrain_optimizers):
    optimizer.zero_grad()
    init_states = model.init_states(inputs.shape[0], inputs.device)
    # deterministic=False enables dropout (matches Flax deterministic=False path)
    with _lstm_autocast(inputs.device):
      logits, _ = model(inputs, init_states, return_sequence=True, deterministic=False)
    loss = F.cross_entropy(
        logits.reshape(-1, model.vocab_size),
        targets.reshape(-1),
        reduction='mean',
    )
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 4.0)
    # Match the JAX behavior: Adam with lr=1.0 and an explicit scaling by current_lr each step.
    # Setting param_groups[*]['lr'] = current_lr produces the same applied update for Adam.
    for g in optimizer.param_groups:
      g['lr'] = current_lr
    optimizer.step()
    losses.append(loss.item())
  return float(np.mean(losses))


def process(compress, length, vocab_size, coder, data):
  """Main processing loop for compression and decompression.

  Streams characters through the LSTM ensemble, periodically retrains on recent
  history, and drives the arithmetic coder one symbol at a time.

  When ``model_type == "transformer_xl"``, dispatches to the NNCP-style
  streaming loop in ``_process_transformer_xl`` instead. When ``model_type ==
  "hybrid"``, dispatches to the LSTM + Transformer-XL ensemble loop in
  ``_process_hybrid``.
  """
  _mt = globals().get('model_type', 'lstm')
  if _mt == 'transformer_xl':
    return _process_transformer_xl(compress, length, vocab_size, coder, data)
  if _mt == 'hybrid':
    return _process_hybrid(compress, length, vocab_size, coder, data)
  start = time.time()
  last_print_time = start
  reset_seed()

  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  # Optional TensorBoard logging (controlled by global params `tensorboard`,
  # `tensorboard_run_name`, and `tensorboard_logdir`).
  tb_writer = None
  if globals().get('tensorboard', False):
    from torch.utils.tensorboard import SummaryWriter
    _tb_logdir = globals().get('tensorboard_logdir', 'data/tensorboard')
    _tb_run = globals().get('tensorboard_run_name', 'torch_compress')
    tb_writer = SummaryWriter(log_dir=os.path.join(_tb_logdir, _tb_run))
    tb_writer.add_text('config', (
        f"backend=torch device={device} "
        f"batch_size={batch_size} seq_length={seq_length} "
        f"rnn_units={rnn_units} num_layers={num_layers} "
        f"embedding_size={embedding_size} ensemble_size={ensemble_size}"
    ))
    print(f"[tensorboard] writing to {tb_writer.log_dir}")

  lr_schedule = parse_schedule(learning_rate_schedule)
  retrain_p_schedule = parse_schedule(retrain_period_schedule)
  retrain_l_schedule = parse_schedule(retrain_lr_schedule)

  _use_bf16_active = bool(globals().get('use_bf16', False)) and torch.cuda.is_available()
  print(f"precision={'bf16' if _use_bf16_active else 'fp32'}")
  print(f"batch_size={batch_size}, seq_length={seq_length}, rnn_units={rnn_units}, num_layers={num_layers}, "
        f"embedding_size={embedding_size}, ensemble_size={ensemble_size}, learning_rate_schedule={learning_rate_schedule}, "
        f"adam_b1={adam_b1}, adam_b2={adam_b2}, adam_eps={adam_eps}, "
        f"retrain_period_schedule={retrain_period_schedule}, "
        f"retrain_block_len={retrain_block_len}, retrain_seq_length={retrain_seq_length}, "
        f"retrain_batch_size={retrain_batch_size}, retrain_lr_schedule={retrain_lr_schedule}, "
        f"retrain_dropout={retrain_dropout}, total_parts={total_parts}, current_part={current_part}")

  # Build ensemble
  models = []
  optimizers = []
  retrain_optimizers = []
  initial_lr = lr_schedule[0][1] if lr_schedule else 5e-4
  for _ in range(ensemble_size):
    m = build_model(globals().get('model_type', 'lstm'),
                    vocab_size=vocab_size, embedding_size=embedding_size,
                    rnn_units=rnn_units, num_layers=num_layers,
                    dropout_rate=retrain_dropout).to(device)
    models.append(m)
    optimizers.append(torch.optim.Adam(
        m.parameters(), lr=initial_lr, betas=(adam_b1, adam_b2), eps=adam_eps))
    retrain_optimizers.append(torch.optim.Adam(
        m.parameters(), lr=1.0, betas=(adam_b1, adam_b2), eps=adam_eps))

  # Restore model weights if a checkpoint exists
  ckpt_dir = os.path.abspath('data/ckpt')
  if checkpoint:
    ckpt_path = os.path.join(ckpt_dir, 'checkpoint.pt')
    if os.path.exists(ckpt_path):
      ckpt = torch.load(ckpt_path, map_location=device)
      for i, m in enumerate(models):
        m.load_state_dict(ckpt['models'][i])
      print("Restored model weights from checkpoint.")
    else:
      print("No checkpoint found. Starting from scratch.")

  total_params = sum(p.numel() for p in models[0].parameters()) * ensemble_size
  print("\n" + "=" * 80)
  print(f"Model Architecture (Ensemble Size: {ensemble_size})")
  print("=" * 80)
  print(models[0])
  print(f"\nTotal Ensemble Parameters: {total_params:,}")
  print("=" * 80 + "\n")

  split = math.ceil(length / batch_size)
  part_len = math.ceil(split / total_parts)
  start_pos = (current_part - 1) * part_len
  end_pos = min(start_pos + part_len, split)
  print(f"Processing part {current_part} of {total_parts}. Step range: {start_pos} to {end_pos - 1} (Total split: {split})")

  # Uniform prior used during the very first symbol of each stream.
  freq = np.cumsum(np.full(vocab_size, (1.0 / vocab_size)) * 10000000 + 1)

  pos = 0

  def fresh_states():
    return [m.init_states(batch_size, device) for m in models]

  model_state_loaded = False
  if start_pos > 0 and checkpoint:
    model_state_path = os.path.join(ckpt_dir, 'model_state.pt')
    if os.path.exists(model_state_path):
      print("Attempting to load model states from checkpoint...")
      try:
        loaded = torch.load(model_state_path, map_location=device)
        seq_input = loaded['seq_input'].to(device)
        if seq_input.shape != (batch_size, seq_length):
          raise ValueError(
              f"Loaded seq_input shape {tuple(seq_input.shape)} mismatch with batch/seq params.")
        states_queue = [
            [[(h.to(device), c.to(device)) for (h, c) in layer_states]
             for layer_states in step_states]
            for step_states in loaded['states_queue']
        ]
        if 'opt_states' in loaded:
          for opt, sd in zip(optimizers, loaded['opt_states']):
            opt.load_state_dict(sd)
          print("Restored main optimizer state.")
        if 'retrain_opt_states' in loaded:
          for opt, sd in zip(retrain_optimizers, loaded['retrain_opt_states']):
            opt.load_state_dict(sd)
          print("Restored retraining optimizer state.")
        print("Successfully loaded model states. Skipping warmup.")
        model_state_loaded = True
        pos = start_pos
      except Exception as e:
        print(f"Failed to load model states: {e}. Falling back to warmup.")

  if start_pos > 0 and not model_state_loaded:
    warmup_len = 500
    run_start_pos = max(0, start_pos - warmup_len)

    w_symbols = []
    for i in range(batch_size):
      start_idx_window = run_start_pos - seq_length + 1
      batch_syms = []
      for k in range(seq_length):
        idx_k = start_idx_window + k
        s_idx = idx_k + i * split
        if s_idx < 0:
          val = data[i * split] if i * split < len(data) else 0
        else:
          val = data[s_idx] if s_idx < len(data) else 0
        batch_syms.append(val)
      w_symbols.append(batch_syms)
    seq_input = torch.tensor(w_symbols, dtype=torch.long, device=device)

    states_queue = [fresh_states() for _ in range(seq_length)]

    print(f"Warming up states from {run_start_pos} to {start_pos}...")
    for w_pos in range(run_start_pos, start_pos):
      state_in = states_queue.pop(0)
      with torch.no_grad():
        new_state_per_model = []
        for model, states in zip(models, state_in):
          with _lstm_autocast(seq_input.device):
            _, ns = model(seq_input, states, return_sequence=False, deterministic=True)
          new_state_per_model.append(ns)
      states_queue.append(detach_states(new_state_per_model))

      next_in_symbols = []
      for i in range(batch_size):
        idx = w_pos + 1 + i * split
        val = data[idx] if idx < len(data) else 0
        next_in_symbols.append(val)
      next_in_t = torch.tensor(next_in_symbols, dtype=torch.long, device=device).unsqueeze(1)
      seq_input = torch.cat([seq_input[:, 1:], next_in_t], dim=1)

    pos = start_pos

  elif not model_state_loaded:
    initial_symbols = []
    for i in range(batch_size):
      initial_symbols.append(get_symbol(i * split, length, freq, coder, compress, data))
    seq_input = torch.tensor(initial_symbols, dtype=torch.long, device=device).unsqueeze(1).repeat(1, seq_length)
    states_queue = [fresh_states() for _ in range(seq_length)]
    pos = 0

  cross_entropy = 0.0
  denom = 0.0
  template = '{:0.2f}%\tcross entropy: {:0.2f}\ttime: {:0.2f}\tlr: {:0.8f}\tstep: {}'

  last_retrain_pos = 0
  current_lr = initial_lr

  while pos < end_pos:
    # Update online learning rate
    current_lr = get_scheduled_value(lr_schedule, pos)
    for opt in optimizers:
      for g in opt.param_groups:
        g['lr'] = current_lr

    current_retrain_period = get_scheduled_value(retrain_p_schedule, pos)
    current_retrain_lr = get_scheduled_value(retrain_l_schedule, pos)

    if current_retrain_period > 0 and (pos - last_retrain_pos) >= current_retrain_period:
      retrain_start_time = time.time()

      r_start_step = max(0, pos - retrain_block_len)
      r_end_step = pos

      all_inputs = []
      all_targets = []

      r_step = r_start_step
      while r_step < r_end_step:
        for i in range(batch_size):
          base_idx = r_step + i * split
          start_idx = base_idx
          end_idx = start_idx + retrain_seq_length + 1
          current_stream_limit = i * split + pos + 1

          stream_segment = data[start_idx: min(end_idx, current_stream_limit)]
          if len(stream_segment) < retrain_seq_length + 1:
            stream_segment = list(stream_segment) + [0] * (retrain_seq_length + 1 - len(stream_segment))

          all_inputs.append(stream_segment[:-1])
          all_targets.append(stream_segment[1:])

        r_step += retrain_seq_length

      all_inputs_t = torch.tensor(all_inputs, dtype=torch.long, device=device)
      all_targets_t = torch.tensor(all_targets, dtype=torch.long, device=device)

      total_examples = all_inputs_t.shape[0]
      remainder = total_examples % retrain_batch_size
      if remainder != 0:
        all_inputs_t = all_inputs_t[:-remainder]
        all_targets_t = all_targets_t[:-remainder]
        total_examples -= remainder

      print(f"Starting retraining at step {pos}... (period={current_retrain_period:.1f}, "
            f"lr={current_retrain_lr:.8f}, examples={total_examples})")

      retrain_loss_sum = 0.0
      retrain_batches = 0
      if total_examples > 0:
        for i in range(0, total_examples, retrain_batch_size):
          batch_inputs = all_inputs_t[i: i + retrain_batch_size]
          batch_targets = all_targets_t[i: i + retrain_batch_size]
          rl = retrain_step(models, retrain_optimizers, batch_inputs, batch_targets, current_retrain_lr)
          retrain_loss_sum += rl
          retrain_batches += 1

      last_retrain_pos = pos
      retrain_duration = time.time() - retrain_start_time
      print(f"Retraining finished. Duration: {retrain_duration:.2f}s")
      if tb_writer is not None and retrain_batches > 0:
        tb_writer.add_scalar('retrain/loss', retrain_loss_sum / retrain_batches, pos)
        tb_writer.add_scalar('retrain/lr', current_retrain_lr, pos)
        tb_writer.add_scalar('retrain/duration_sec', retrain_duration, pos)

    state_in = states_queue.pop(0)

    # Forward pass (autograd enabled, dropout disabled to match deterministic=True)
    probs_np, logits_list, new_states = forward_ensemble(models, seq_input, state_in)

    # Drive the arithmetic coder symbol-by-symbol
    current_symbols = []
    current_mask = []
    for i in range(batch_size):
      p_i = probs_np[i]
      freq_i = np.cumsum(p_i * 10000000 + 1)
      index = pos + 1 + i * split
      symbol = get_symbol(index, length, freq_i, coder, compress, data)
      current_symbols.append(symbol)
      current_mask.append(1.0 if index < length else 0.0)

    symbols_t = torch.tensor(current_symbols, dtype=torch.long, device=device)
    mask_t = torch.tensor(current_mask, dtype=torch.float32, device=device)

    loss_val, loss_denom = backward_and_step(
        logits_list, optimizers, models, symbols_t, mask_t, vocab_size)

    cross_entropy += loss_val
    denom += loss_denom

    states_queue.append(detach_states(new_states))

    seq_input = torch.cat([seq_input[:, 1:], symbols_t.unsqueeze(1)], dim=1)
    pos += 1

    if time.time() - last_print_time >= 20:
      last_print_time = time.time()
      time_diff = last_print_time - start
      current_bpc = (cross_entropy / denom) / np.log(2)
      print(template.format(pos / split * 100, current_bpc, time_diff, current_lr, pos))
      if tb_writer is not None:
        tb_writer.add_scalar('train/bpc', current_bpc, pos)
        tb_writer.add_scalar('train/cross_entropy_total', cross_entropy, pos)
        tb_writer.add_scalar('train/lr', current_lr, pos)
        tb_writer.add_scalar('train/elapsed_sec', time_diff, pos)
        tb_writer.add_scalar('train/steps_per_sec', pos / time_diff if time_diff > 0 else 0.0, pos)

  if checkpoint:
    print("Saving checkpoint...")
    if not os.path.exists(ckpt_dir):
      os.makedirs(ckpt_dir)

    torch.save({
        'models': [m.state_dict() for m in models],
        'opt_states': [o.state_dict() for o in optimizers],
        'retrain_opt_states': [o.state_dict() for o in retrain_optimizers],
    }, os.path.join(ckpt_dir, 'checkpoint.pt'))
    print("Checkpoint saved.")

    ac_state = {
        'coder': coder.get_state(),
        'bitstream': coder.output.get_state() if compress else coder.input.get_state(),
    }
    if not compress:
      ac_state['file_pos'] = coder.input.input.tell()
    with open(os.path.join(ckpt_dir, 'ac_state.pkl'), 'wb') as f:
      pickle.dump(ac_state, f)
    print("AC state saved.")

    seq_input_cpu = seq_input.detach().cpu()
    states_queue_cpu = [
        [[(h.detach().cpu(), c.detach().cpu()) for (h, c) in layer_states]
         for layer_states in step_states]
        for step_states in states_queue
    ]
    torch.save({
        'seq_input': seq_input_cpu,
        'states_queue': states_queue_cpu,
        'opt_states': [o.state_dict() for o in optimizers],
        'retrain_opt_states': [o.state_dict() for o in retrain_optimizers],
    }, os.path.join(ckpt_dir, 'model_state.pt'))
    print("Model state (including optimizers) saved.")

  if tb_writer is not None:
    final_time = time.time() - start
    if denom > 0:
      tb_writer.add_scalar('train/bpc', (cross_entropy / denom) / np.log(2), pos)
    tb_writer.add_scalar('train/elapsed_sec', final_time, pos)
    tb_writer.add_text('summary', (
        f"final_pos={pos} elapsed_sec={final_time:.2f} "
        f"final_bpc={(cross_entropy / denom) / np.log(2) if denom > 0 else float('nan'):.4f}"
    ))
    tb_writer.flush()
    tb_writer.close()


def _process_transformer_xl(compress, length, vocab_size, coder, data):
  """Transformer-XL streaming compress/decompress, modeled on NNCP v2's train().

  Differences from the LSTM path:
  - Mems carry forward naturally between steps; there is no state queue / BPTT.
  - Each step feeds (ext_tgt_len + 1) tokens and predicts the last token.
    (Phase 2A runs with tgt_len=1 only — multi-target packing is Phase 2B.)
  - Online dropout is off (deterministic=True), matching the LSTM path.

  Online retraining honors ``retrain_period_schedule`` and runs in NNCP's
  retrain shape (``retrain_tgt_len``, ``retrain_mem_len``) via
  ``_retrain_transformer_xl``; ``model.reset_length`` toggles around the call.

  Mixed-precision: when ``use_bf16`` is True, forward + backward run under
  ``torch.autocast`` with bfloat16 activations while the model parameters and
  optimizer state stay in fp32 (standard mixed-precision pattern). Logits are
  cast back to fp32 before the AC encoder consumes them so the arithmetic
  coding maths is identical to the fp32 path. On GH200 / H100 this is ~10-15x
  faster than fp32 for the large NNCP-base config; on CPU it has no
  meaningful effect.

  Limitations (later work):
  - Multi-part / checkpointing not yet supported (asserts current_part == 1).
  """
  assert current_part == 1 and total_parts == 1, (
      "transformer_xl backend currently supports only single-part runs."
  )
  if checkpoint:
    print("[transformer_xl] Warning: checkpoint=True is not yet supported; ignoring.")

  # bf16 mixed precision is round-trip safe given the
  # allow_bf16_reduced_precision_reduction = False flag set in IMPORTS_SRC.
  # Without that flag, bf16 matmul kernels accumulated reductions in bf16
  # with non-deterministic thread ordering, breaking encode/decode agreement
  # on probabilities during retrain (see commit 3f6a620 for the diagnostic).
  use_bf16_flag = bool(globals().get('use_bf16', False))
  ac_dtype = torch.bfloat16 if use_bf16_flag else torch.float32

  start = time.time()
  last_print_time = start
  reset_seed()
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  tb_writer = None
  if globals().get('tensorboard', False):
    from torch.utils.tensorboard import SummaryWriter
    _tb_logdir = globals().get('tensorboard_logdir', 'data/tensorboard')
    _tb_run = globals().get('tensorboard_run_name', 'torch_compress')
    tb_writer = SummaryWriter(log_dir=os.path.join(_tb_logdir, _tb_run))
    tb_writer.add_text('config', (
        f"backend=torch model=transformer_xl device={device} "
        f"batch_size={batch_size} ext_tgt_len={ext_tgt_len} mem_len={mem_len} "
        f"n_layer={n_layer} d_model={d_model} n_head={n_head} d_head={d_head} "
        f"d_inner={d_inner} ensemble_size={ensemble_size}"
    ))
    print(f"[tensorboard] writing to {tb_writer.log_dir}")

  # Transformer-XL uses its own LR schedules (NNCP-base defaults). The LSTM-
  # tuned `learning_rate_schedule` / `retrain_lr_schedule` stay reserved for
  # the LSTM path -- transformers want lower LRs and a different decay shape.
  lr_schedule_str = globals().get('learning_rate_schedule_xl', learning_rate_schedule)
  lr_schedule = parse_schedule(lr_schedule_str)

  print(f"[transformer_xl] batch_size={batch_size}, ext_tgt_len={ext_tgt_len}, "
        f"mem_len={mem_len}, n_layer={n_layer}, d_model={d_model}, n_head={n_head}, "
        f"d_head={d_head}, d_inner={d_inner}, ensemble_size={ensemble_size}, "
        f"learning_rate_schedule_xl={lr_schedule_str}, "
        f"adam_b1={adam_b1}, adam_b2={adam_b2}, adam_eps={adam_eps}")

  models = []
  optimizers = []
  retrain_optimizers = []
  initial_lr = lr_schedule[0][1] if lr_schedule else 5e-4
  # NNCP-aligned Adam epsilon (1e-9 vs torch_compress's tuned 1e-12 for LSTM).
  _adam_eps_xl = float(globals().get('adam_eps_xl', 1e-9))
  for _ in range(ensemble_size):
    m = build_model('transformer_xl',
                    vocab_size=vocab_size, embedding_size=embedding_size,
                    rnn_units=rnn_units, num_layers=num_layers,
                    dropout_rate=0.0).to(device)
    models.append(m)
    optimizers.append(torch.optim.Adam(
        m.parameters(), lr=initial_lr, betas=(adam_b1, adam_b2), eps=_adam_eps_xl))
    # Separate optimizer state for retraining, mirroring NNCP's saved-state
    # toggle and torch_compress's two-optimizer pattern. lr is overwritten
    # per-step inside the retrain function.
    retrain_optimizers.append(torch.optim.Adam(
        m.parameters(), lr=1.0, betas=(adam_b1, adam_b2), eps=_adam_eps_xl))

  total_params = sum(p.numel() for p in models[0].parameters()) * ensemble_size
  print("\n" + "=" * 80)
  print(f"Transformer-XL Architecture (Ensemble Size: {ensemble_size})")
  print("=" * 80)
  print(models[0])
  print(f"\nTotal Ensemble Parameters: {total_params:,}")
  print("=" * 80 + "\n")

  split = math.ceil(length / batch_size)
  end_pos = split
  print(f"Processing single part. Stream length per batch: {split}")

  # Uniform prior used to AC-code the very first symbol of each parallel stream.
  freq = np.cumsum(np.full(vocab_size, (1.0 / vocab_size)) * 10000000 + 1)

  qlen = ext_tgt_len + 1  # tgt_len = 1 for streaming inference

  initial_symbols = []
  for i in range(batch_size):
    initial_symbols.append(get_symbol(i * split, length, freq, coder, compress, data))
  # Bootstrap input window: the just-coded symbol replicated qlen times.
  # Real context accumulates as the stream advances.
  seq_input = (
      torch.tensor(initial_symbols, dtype=torch.long, device=device)
      .unsqueeze(1)
      .repeat(1, qlen)
  )
  mems_per_model = _xl_init_mems(models, batch_size, device, ac_dtype)

  retrain_p_schedule = parse_schedule(retrain_period_schedule)
  retrain_l_schedule_str = globals().get('retrain_lr_schedule_xl', retrain_lr_schedule)
  retrain_l_schedule = parse_schedule(retrain_l_schedule_str)

  cross_entropy = 0.0
  denom = 0.0
  template = '{:0.2f}%\tcross entropy: {:0.2f}\ttime: {:0.2f}\tlr: {:0.8f}\tstep: {}'

  pos = 0
  current_lr = initial_lr
  last_retrain_pos = 0

  while pos < end_pos:
    current_lr = get_scheduled_value(lr_schedule, pos)
    for opt in optimizers:
      for g in opt.param_groups:
        g['lr'] = current_lr

    # NNCP-style retrain: reshape model to retrain_(tgt_len, mem_len), run a
    # streaming pass over the trailing retrain_block_len chars with mems
    # carrying within the pass, then restore the streaming shape.
    current_retrain_period = get_scheduled_value(retrain_p_schedule, pos)
    current_retrain_lr = get_scheduled_value(retrain_l_schedule, pos)
    if current_retrain_period > 0 and (pos - last_retrain_pos) >= current_retrain_period:
      retrain_start_time = time.time()
      retrain_loss = _retrain_transformer_xl(
          models=models,
          retrain_optimizers=retrain_optimizers,
          current_lr=current_retrain_lr,
          file_data=data,
          file_pos=pos + 1,
          retrain_block_len=retrain_block_len,
          retrain_tgt_len=retrain_tgt_len,
          retrain_mem_len=retrain_mem_len,
          retrain_batch_size=retrain_batch_size,
          stream_mem_len=mem_len,
          vocab_size=vocab_size,
          device=device,
      )
      last_retrain_pos = pos
      retrain_duration = time.time() - retrain_start_time
      print(f"[transformer_xl] retrain done at step {pos}: "
            f"loss={retrain_loss:.4f}, duration={retrain_duration:.2f}s")
      if tb_writer is not None:
        tb_writer.add_scalar('retrain/loss', retrain_loss, pos)
        tb_writer.add_scalar('retrain/lr', current_retrain_lr, pos)
        tb_writer.add_scalar('retrain/duration_sec', retrain_duration, pos)
      # Mems were attached to the pre-retrain stream-shape model; the retrain
      # mutated weights and toggled reset_length back to streaming, but the
      # mems still reflect activations from before the weight update. Reset
      # them to empty so the next forward pass re-builds context with the
      # updated weights.
      mems_per_model = _xl_init_mems(models, batch_size, device, ac_dtype)

    # Forward — autograd ON, dropout off (matches LSTM forward_ensemble path).
    log_probs_list = []
    logits_list = []
    new_mems_list = []
    for model, mems in zip(models, mems_per_model):
      with torch.autocast(device_type=device.type, dtype=ac_dtype, enabled=use_bf16_flag):
        logits, new_mems = model(seq_input, mems, return_sequence=False, deterministic=True)
      # AC consumes fp32 probabilities; cast logits up so the loss + AC math
      # below is fp32 even when the forward ran in bf16.
      logits = logits.float()
      logits_list.append(logits)
      log_probs_list.append(F.log_softmax(logits, dim=-1))
      new_mems_list.append(new_mems)
    mean_log_probs = torch.stack(log_probs_list, dim=0).mean(dim=0)
    probs = F.softmax(mean_log_probs, dim=-1).detach().cpu().numpy()

    # Drive the arithmetic coder: one symbol per parallel stream.
    current_symbols = []
    current_mask = []
    for i in range(batch_size):
      freq_i = np.cumsum(probs[i] * 10000000 + 1)
      index = pos + 1 + i * split
      symbol = get_symbol(index, length, freq_i, coder, compress, data)
      current_symbols.append(symbol)
      current_mask.append(1.0 if index < length else 0.0)

    symbols_t = torch.tensor(current_symbols, dtype=torch.long, device=device)
    mask_t = torch.tensor(current_mask, dtype=torch.float32, device=device)

    # Reuse the LSTM-side helper — same per-step loss/clip/step shape.
    # NNCP-aligned: pass clip_xl (default 0.25) instead of the LSTM's 4.0.
    loss_val, loss_denom = backward_and_step(
        logits_list, optimizers, models, symbols_t, mask_t, vocab_size,
        clip=float(globals().get('clip_xl', 0.25)))
    cross_entropy += loss_val
    denom += loss_denom

    # Carry mems forward (already detached inside MemTransformerLM._update_mems).
    mems_per_model = new_mems_list
    # Slide window: drop oldest, append the symbol just coded.
    seq_input = torch.cat([seq_input[:, 1:], symbols_t.unsqueeze(1)], dim=1)
    pos += 1

    if time.time() - last_print_time >= 20:
      last_print_time = time.time()
      time_diff = last_print_time - start
      current_bpc = (cross_entropy / denom) / np.log(2)
      print(template.format(pos / split * 100, current_bpc, time_diff, current_lr, pos))
      if tb_writer is not None:
        tb_writer.add_scalar('train/bpc', current_bpc, pos)
        tb_writer.add_scalar('train/cross_entropy_total', cross_entropy, pos)
        tb_writer.add_scalar('train/lr', current_lr, pos)
        tb_writer.add_scalar('train/elapsed_sec', time_diff, pos)
        tb_writer.add_scalar('train/steps_per_sec', pos / time_diff if time_diff > 0 else 0.0, pos)

  if tb_writer is not None:
    final_time = time.time() - start
    if denom > 0:
      tb_writer.add_scalar('train/bpc', (cross_entropy / denom) / np.log(2), pos)
    tb_writer.add_scalar('train/elapsed_sec', final_time, pos)
    tb_writer.add_text('summary', (
        f"final_pos={pos} elapsed_sec={final_time:.2f} "
        f"final_bpc={(cross_entropy / denom) / np.log(2) if denom > 0 else float('nan'):.4f}"
    ))
    tb_writer.flush()
    tb_writer.close()


def _xl_init_mems(models, batch_size, device, ac_dtype):
  """Build empty mems for each ensemble member, casting to ``ac_dtype``.

  ``MemTransformerLM.init_mems`` returns tensors of the model's parameter
  dtype (fp32). Under bf16 mixed-precision the first forward pass produces
  bf16 hidden states; ``_update_mems`` then concatenates the (fp32) initial
  empty mems with the (bf16) hids, which would error on dtype mismatch in
  ``torch.cat``. Pre-casting here keeps the mems in the right dtype from the
  start, and is a no-op for fp32.
  """
  out = []
  for m in models:
    mems = m.init_states(batch_size, device)
    if mems is None:
      out.append(None)
      continue
    if ac_dtype is not None and ac_dtype != torch.float32:
      mems = [t.to(ac_dtype) for t in mems]
    out.append(mems)
  return out


def _retrain_transformer_xl(*, models, retrain_optimizers, current_lr, file_data,
                            file_pos, retrain_block_len, retrain_tgt_len,
                            retrain_mem_len, retrain_batch_size, stream_mem_len,
                            vocab_size, device):
  """NNCP-style retraining pass on the trailing ``retrain_block_len`` chars.

  Mirrors NNCP v2's ``retrain()`` from ``nncp.py``:

  - ``model.reset_length(retrain_tgt_len, 0, retrain_mem_len)`` reshapes the
    model for retrain (typically tgt_len=64, mem_len=128 for NNCP-base).
  - The retrain window is split into ``retrain_batch_size`` parallel streams,
    each ``block_stride = block_len // retrain_batch_size`` chars long.
  - Steps process ``retrain_tgt_len`` tokens at a time; mems carry forward
    *within* the retrain pass (this is the part that distinguishes NNCP-style
    retrain from the LSTM-style fresh-state-per-batch retrain).
  - Dropout is on (``deterministic=False`` -> ``model.train()`` inside the
    adapter); the model's dropout rate was set at construction via the
    ``dropout`` notebook hparam.
  - On the very first step, the input row is dummied (matches NNCP exactly --
    avoids the off-by-one negative index at stream_pos=0).
  - At exit, ``model.reset_length(1, 0, stream_mem_len)`` restores the
    streaming-inference shape so the caller's ``mems_per_model`` rebuild
    (which the caller does immediately after) produces correctly-sized mems.

  Returns the mean per-step cross-entropy loss across the ensemble (in nats).
  """
  block_len = min(file_pos, retrain_block_len)
  if block_len < retrain_batch_size * retrain_tgt_len:
    return 0.0  # not enough trailing data for a single retrain step
  block_start = file_pos - block_len
  block_stride = block_len // retrain_batch_size
  if block_stride < retrain_tgt_len:
    return 0.0

  # Switch every ensemble member to retrain shape before the first forward.
  for model in models:
    model.reset_length(retrain_tgt_len, 0, retrain_mem_len)

  use_bf16_flag = bool(globals().get('use_bf16', False))
  ac_dtype = torch.bfloat16 if use_bf16_flag else torch.float32

  ensemble_losses = []
  try:
    for model, optimizer in zip(models, retrain_optimizers):
      mems = model.init_states(retrain_batch_size, device)
      if mems is not None and ac_dtype != torch.float32:
        mems = [t.to(ac_dtype) for t in mems]
      stream_pos = 0
      step_losses = []
      while (stream_pos + retrain_tgt_len) <= block_stride:
        data0 = []
        target0 = []
        for j in range(retrain_tgt_len):
          target_pos = block_start + stream_pos + j
          target_row = file_data[target_pos: target_pos + block_stride * retrain_batch_size: block_stride]
          target0.append(target_row)
          if stream_pos == 0:
            data_row = [0] * retrain_batch_size  # dummy first step (matches NNCP)
          else:
            input_pos = target_pos - 1
            data_row = file_data[input_pos: input_pos + block_stride * retrain_batch_size: block_stride]
          data0.append(data_row)

        # Build (retrain_tgt_len, retrain_batch_size) tensors then transpose
        # to (batch, tgt_len) to match the adapter's batch-first interface.
        data_t = torch.tensor(data0, dtype=torch.long, device=device).t().contiguous()
        target_t = torch.tensor(target0, dtype=torch.long, device=device).t().contiguous()

        optimizer.zero_grad()
        # return_sequence=True -> logits at every position; deterministic=False
        # -> dropout on (model.train() set by the adapter).
        with torch.autocast(device_type=device.type, dtype=ac_dtype, enabled=use_bf16_flag):
          logits, mems = model(data_t, mems, return_sequence=True, deterministic=False)
        # Cast to fp32 so cross_entropy + backward run in fp32 (matches the
        # streaming loop's pattern).
        logits = logits.float()
        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            target_t.reshape(-1),
            reduction='mean',
        )
        loss.backward()
        # NNCP-aligned clip (0.25) for the transformer retrain pass.
        torch.nn.utils.clip_grad_norm_(model.parameters(),
            float(globals().get('clip_xl', 0.25)))
        for g in optimizer.param_groups:
          g['lr'] = current_lr
        optimizer.step()
        step_losses.append(loss.item())
        stream_pos += retrain_tgt_len

      ensemble_losses.append(float(np.mean(step_losses)) if step_losses else 0.0)
  finally:
    # Always restore the streaming shape, even on exception.
    for model in models:
      model.reset_length(1, 0, stream_mem_len)

  return float(np.mean(ensemble_losses)) if ensemble_losses else 0.0


def _process_hybrid(compress, length, vocab_size, coder, data):
  """LSTM + Transformer-XL hybrid streaming loop with geometric-mean ensembling.

  At every step both submodels forward against the same just-coded symbol
  context. AC sees ``softmax(0.5 * (log_softmax(lstm_logits) +
  log_softmax(xl_logits)))`` -- the geometric mean of the two distributions.
  Each submodel backprops independently against the actual symbol with its
  own optimizer (and its own LR schedule).

  Per-submodel state management mirrors each model's solo loop:
    - LSTM: states_queue of seq_length snapshots, BPTT replay through the
      window, push detached state after the forward.
    - Transformer-XL: mems list, carries forward via _update_mems inside the
      model. Pre-cast to bf16 if use_bf16; reset to empty after retrain.

  Limitations:
  - ensemble_size > 1 is rejected here -- each "ensemble member" is itself a
    hybrid pair, and stacking multiple hybrids would double-count parameters
    in unhelpful ways. Use a separate run with model_type="lstm" or
    "transformer_xl" if you want to homogeneous-ensemble.
  - Multi-part / checkpointing not supported (asserts current_part == 1).
  """
  assert current_part == 1 and total_parts == 1, (
      "hybrid backend currently supports only single-part runs."
  )
  assert ensemble_size == 1, (
      "hybrid backend currently supports only ensemble_size=1; the hybrid "
      "itself is already a 2-model ensemble."
  )
  if checkpoint:
    print("[hybrid] Warning: checkpoint=True is not yet supported; ignoring.")

  # bf16 mixed precision: applies only to the Transformer-XL submodel's
  # forward (LSTM cell loop runs in fp32 regardless -- bf16 isn't wired into
  # LSTMModel and the BPTT compounding makes it risky there).
  use_bf16_flag = bool(globals().get('use_bf16', False))
  ac_dtype = torch.bfloat16 if use_bf16_flag else torch.float32

  start = time.time()
  last_print_time = start
  reset_seed()
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  tb_writer = None
  if globals().get('tensorboard', False):
    from torch.utils.tensorboard import SummaryWriter
    _tb_logdir = globals().get('tensorboard_logdir', 'data/tensorboard')
    _tb_run = globals().get('tensorboard_run_name', 'torch_compress')
    tb_writer = SummaryWriter(log_dir=os.path.join(_tb_logdir, _tb_run))
    tb_writer.add_text('config', (
        f"backend=torch model=hybrid device={device} "
        f"batch_size={batch_size} seq_length={seq_length} "
        f"rnn_units={rnn_units} num_layers={num_layers} "
        f"embedding_size={embedding_size} ext_tgt_len={ext_tgt_len} "
        f"mem_len={mem_len} n_layer={n_layer} d_model={d_model}"
    ))

  # Two LR schedules: LSTM uses learning_rate_schedule (its tuned default);
  # Transformer-XL uses learning_rate_schedule_xl (NNCP-base default).
  lstm_lr_schedule = parse_schedule(learning_rate_schedule)
  xl_lr_schedule_str = globals().get('learning_rate_schedule_xl', learning_rate_schedule)
  xl_lr_schedule = parse_schedule(xl_lr_schedule_str)
  retrain_p_schedule = parse_schedule(retrain_period_schedule)
  retrain_l_schedule = parse_schedule(retrain_lr_schedule)
  retrain_l_schedule_xl_str = globals().get('retrain_lr_schedule_xl', retrain_lr_schedule)
  retrain_l_schedule_xl = parse_schedule(retrain_l_schedule_xl_str)

  initial_lstm_lr = lstm_lr_schedule[0][1] if lstm_lr_schedule else 5e-4
  initial_xl_lr = xl_lr_schedule[0][1] if xl_lr_schedule else 7.9e-5

  # Build hybrid model + per-submodel optimizers.
  model = build_model('hybrid', vocab_size=vocab_size, embedding_size=embedding_size,
                      rnn_units=rnn_units, num_layers=num_layers,
                      dropout_rate=retrain_dropout).to(device)

  # LSTM half keeps torch_compress's tuned adam_eps; Transformer-XL half
  # uses NNCP-aligned adam_eps_xl. Mixer keeps adam_eps (the LSTM-side value
  # is fine for a tiny gating MLP).
  _adam_eps_xl = float(globals().get('adam_eps_xl', 1e-9))
  lstm_opt = torch.optim.Adam(
      model.lstm.parameters(), lr=initial_lstm_lr, betas=(adam_b1, adam_b2), eps=adam_eps)
  xl_opt = torch.optim.Adam(
      model.transformer_xl.parameters(), lr=initial_xl_lr, betas=(adam_b1, adam_b2), eps=_adam_eps_xl)
  # Separate retrain optimizer state per submodel (lr=1.0 placeholder; per-step set).
  lstm_retrain_opt = torch.optim.Adam(
      model.lstm.parameters(), lr=1.0, betas=(adam_b1, adam_b2), eps=adam_eps)
  xl_retrain_opt = torch.optim.Adam(
      model.transformer_xl.parameters(), lr=1.0, betas=(adam_b1, adam_b2), eps=_adam_eps_xl)

  # Optional learned mixer optimizer. The mixer is tiny (~300 params) so it
  # can take a much higher LR than the submodels; defaults to 0.01 unless
  # overridden via mixer_lr global.
  mixer_opt = None
  if model.mixer is not None:
    mixer_lr = globals().get('mixer_lr', 0.01)
    mixer_opt = torch.optim.Adam(
        model.mixer.parameters(), lr=mixer_lr, betas=(adam_b1, adam_b2), eps=adam_eps)

  total_params = sum(p.numel() for p in model.parameters())
  lstm_params = sum(p.numel() for p in model.lstm.parameters())
  xl_params = sum(p.numel() for p in model.transformer_xl.parameters())
  mixer_params = sum(p.numel() for p in model.mixer.parameters()) if model.mixer is not None else 0
  print("\n" + "=" * 80)
  print(f"Hybrid LSTM + Transformer-XL" + (" + LearnedMixer" if model.mixer is not None else ""))
  print("=" * 80)
  print(f"LSTM submodel parameters:           {lstm_params:,}")
  print(f"Transformer-XL submodel parameters: {xl_params:,}")
  if model.mixer is not None:
    print(f"LearnedMixer parameters:            {mixer_params:,}")
  print(f"Total parameters:                   {total_params:,}")
  print("=" * 80 + "\n")

  split = math.ceil(length / batch_size)
  end_pos = split

  # Uniform prior used to AC-code the very first symbol of each parallel stream.
  freq = np.cumsum(np.full(vocab_size, (1.0 / vocab_size)) * 10000000 + 1)

  qlen_xl = ext_tgt_len + 1  # Transformer streaming window
  # LSTM streaming window is `seq_length` (same param the LSTM solo path uses).

  initial_symbols = []
  for i in range(batch_size):
    initial_symbols.append(get_symbol(i * split, length, freq, coder, compress, data))

  # Two seq_input windows -- LSTM consumes `seq_length` tokens per step (with
  # BPTT), Transformer-XL consumes `qlen_xl = ext_tgt_len + 1`. They slide in
  # lockstep but can be different widths.
  base = torch.tensor(initial_symbols, dtype=torch.long, device=device).unsqueeze(1)
  lstm_seq_input = base.repeat(1, seq_length)
  xl_seq_input = base.repeat(1, qlen_xl)

  def fresh_lstm_states():
    return model.lstm.init_states(batch_size, device)

  lstm_states_queue = [fresh_lstm_states() for _ in range(seq_length)]

  xl_mems = model.transformer_xl.init_states(batch_size, device)
  if xl_mems is not None and ac_dtype != torch.float32:
    xl_mems = [t.to(ac_dtype) for t in xl_mems]

  cross_entropy = 0.0
  denom = 0.0
  template = '{:0.2f}%\tcross entropy: {:0.2f}\ttime: {:0.2f}\tlstm_lr: {:0.6f}\txl_lr: {:0.6f}\tstep: {}'

  pos = 0
  current_lstm_lr = initial_lstm_lr
  current_xl_lr = initial_xl_lr
  last_retrain_pos = 0

  while pos < end_pos:
    # Per-step LR updates (each submodel uses its own schedule).
    current_lstm_lr = get_scheduled_value(lstm_lr_schedule, pos)
    current_xl_lr = get_scheduled_value(xl_lr_schedule, pos)
    for g in lstm_opt.param_groups: g['lr'] = current_lstm_lr
    for g in xl_opt.param_groups: g['lr'] = current_xl_lr

    # Periodic retrain (both submodels retrain on the same retrain window in
    # their respective shapes). Mems / states are reset after retrain since
    # the weights changed underneath them.
    current_retrain_period = get_scheduled_value(retrain_p_schedule, pos)
    current_lstm_retrain_lr = get_scheduled_value(retrain_l_schedule, pos)
    current_xl_retrain_lr = get_scheduled_value(retrain_l_schedule_xl, pos)
    if current_retrain_period > 0 and (pos - last_retrain_pos) >= current_retrain_period:
      retrain_start_time = time.time()
      _retrain_hybrid(
          model=model,
          lstm_retrain_opt=lstm_retrain_opt,
          xl_retrain_opt=xl_retrain_opt,
          lstm_lr=current_lstm_retrain_lr,
          xl_lr=current_xl_retrain_lr,
          file_data=data,
          file_pos=pos + 1,
          split=split,
          batch_size_streaming=batch_size,
          retrain_block_len=retrain_block_len,
          retrain_seq_length=retrain_seq_length,
          retrain_batch_size_lstm=retrain_batch_size,
          retrain_tgt_len=retrain_tgt_len,
          retrain_mem_len=retrain_mem_len,
          stream_mem_len=mem_len,
          vocab_size=vocab_size,
          device=device,
      )
      last_retrain_pos = pos
      retrain_duration = time.time() - retrain_start_time
      print(f"[hybrid] retrain done at step {pos}: duration={retrain_duration:.2f}s")
      if tb_writer is not None:
        tb_writer.add_scalar('retrain/duration_sec', retrain_duration, pos)
        tb_writer.add_scalar('retrain/lstm_lr', current_lstm_retrain_lr, pos)
        tb_writer.add_scalar('retrain/xl_lr', current_xl_retrain_lr, pos)
      # Reset both submodels' running state -- pre-retrain activations are
      # stale relative to the post-retrain weights.
      lstm_states_queue = [fresh_lstm_states() for _ in range(seq_length)]
      xl_mems = model.transformer_xl.init_states(batch_size, device)
      if xl_mems is not None and ac_dtype != torch.float32:
        xl_mems = [t.to(ac_dtype) for t in xl_mems]

    # ---- Forward both submodels ----
    # LSTM: pop the oldest state from queue, BPTT replay over seq_length
    lstm_state_in = lstm_states_queue.pop(0)
    lstm_logits, new_lstm_state = model.lstm(
        lstm_seq_input, lstm_state_in, return_sequence=False, deterministic=True)

    # Transformer-XL: forward with current mems under autocast
    with torch.autocast(device_type=device.type, dtype=ac_dtype, enabled=use_bf16_flag):
      xl_logits, new_xl_mems = model.transformer_xl(
          xl_seq_input, xl_mems, return_sequence=False, deterministic=True)
    xl_logits = xl_logits.float()

    # ---- Combine: equal-weight or learned-mixer geometric mean ----
    log_probs_lstm = F.log_softmax(lstm_logits, dim=-1)
    log_probs_xl = F.log_softmax(xl_logits, dim=-1)
    if model.mixer is not None:
      # Learned mixer: per-step softmax weights from confidence features.
      # Output is generally unnormalised; log_softmax it to get a valid
      # log-prob distribution for the AC.
      combined_unnorm = model.mixer([log_probs_lstm, log_probs_xl])
      combined_log_probs = F.log_softmax(combined_unnorm, dim=-1)
    else:
      # Equal-weight geometric mean (the original hybrid behaviour).
      combined_log_probs = 0.5 * (log_probs_lstm + log_probs_xl)
    probs = F.softmax(combined_log_probs, dim=-1).detach().cpu().numpy()

    # ---- Drive the AC: one symbol per parallel stream ----
    current_symbols = []
    current_mask = []
    for i in range(batch_size):
      freq_i = np.cumsum(probs[i] * 10000000 + 1)
      index = pos + 1 + i * split
      symbol = get_symbol(index, length, freq_i, coder, compress, data)
      current_symbols.append(symbol)
      current_mask.append(1.0 if index < length else 0.0)

    symbols_t = torch.tensor(current_symbols, dtype=torch.long, device=device)
    mask_t = torch.tensor(current_mask, dtype=torch.float32, device=device)

    # ---- Backward ----
    # Each submodel always gets gradient from its own solo loss (so each
    # learns to be a competent solo predictor regardless of the mixer).
    # If a learned mixer is in play, an additional ensemble-loss term is
    # added: cross-entropy of the mixer's combined distribution against the
    # actual symbol. Gradient from the mixer-loss flows through the mixer
    # AND back through the submodels, so they co-adapt to be good ensemble
    # components in addition to good solos.
    lstm_opt.zero_grad()
    xl_opt.zero_grad()
    if mixer_opt is not None:
      mixer_opt.zero_grad()

    lstm_loss_per = F.cross_entropy(lstm_logits, symbols_t, reduction='none')
    xl_loss_per = F.cross_entropy(xl_logits, symbols_t, reduction='none')
    total_loss = (lstm_loss_per * mask_t).sum() + (xl_loss_per * mask_t).sum()

    if model.mixer is not None:
      # NLL of combined log-probs against actual symbol, masked.
      nll_per = -combined_log_probs.gather(1, symbols_t.unsqueeze(1)).squeeze(1)
      total_loss = total_loss + (nll_per * mask_t).sum()

    total_loss.backward()
    # Per-component clip budgets: LSTM keeps torch_compress's 4.0; the
    # Transformer-XL half uses NNCP-aligned clip_xl (0.25); the mixer is
    # tiny so a generous 4.0 is fine.
    _clip_xl = float(globals().get('clip_xl', 0.25))
    torch.nn.utils.clip_grad_norm_(model.lstm.parameters(), 4.0)
    torch.nn.utils.clip_grad_norm_(model.transformer_xl.parameters(), _clip_xl)
    if model.mixer is not None:
      torch.nn.utils.clip_grad_norm_(model.mixer.parameters(), 4.0)
    lstm_opt.step()
    xl_opt.step()
    if mixer_opt is not None:
      mixer_opt.step()

    # ---- Compute ensemble cross-entropy for logging (against AC's distribution) ----
    # Use the *same* combined_log_probs the AC saw (detached) so the running
    # average matches the realised bpc the AC is producing.
    one_hot = F.one_hot(symbols_t, num_classes=vocab_size).float()
    loss_vals = -(one_hot * combined_log_probs.detach()).sum(dim=-1)
    cross_entropy += (loss_vals * mask_t).sum().item()
    denom += mask_t.sum().item()

    # ---- State carry ----
    # LSTM: detach + push to queue
    lstm_states_queue.append([(h.detach(), c.detach()) for (h, c) in new_lstm_state])
    # Transformer-XL: mems already detached inside _update_mems
    xl_mems = new_xl_mems

    # Slide both windows by the just-coded symbol
    symbols_unsq = symbols_t.unsqueeze(1)
    lstm_seq_input = torch.cat([lstm_seq_input[:, 1:], symbols_unsq], dim=1)
    xl_seq_input = torch.cat([xl_seq_input[:, 1:], symbols_unsq], dim=1)

    pos += 1

    if time.time() - last_print_time >= 20:
      last_print_time = time.time()
      time_diff = last_print_time - start
      current_bpc = (cross_entropy / denom) / np.log(2)
      print(template.format(
          pos / split * 100, current_bpc, time_diff,
          current_lstm_lr, current_xl_lr, pos))
      if tb_writer is not None:
        tb_writer.add_scalar('train/bpc', current_bpc, pos)
        tb_writer.add_scalar('train/cross_entropy_total', cross_entropy, pos)
        tb_writer.add_scalar('train/lstm_lr', current_lstm_lr, pos)
        tb_writer.add_scalar('train/xl_lr', current_xl_lr, pos)
        tb_writer.add_scalar('train/elapsed_sec', time_diff, pos)
        tb_writer.add_scalar('train/steps_per_sec', pos / time_diff if time_diff > 0 else 0.0, pos)

  if tb_writer is not None:
    final_time = time.time() - start
    if denom > 0:
      tb_writer.add_scalar('train/bpc', (cross_entropy / denom) / np.log(2), pos)
    tb_writer.add_scalar('train/elapsed_sec', final_time, pos)
    tb_writer.add_text('summary', (
        f"final_pos={pos} elapsed_sec={final_time:.2f} "
        f"final_bpc={(cross_entropy / denom) / np.log(2) if denom > 0 else float('nan'):.4f}"
    ))
    tb_writer.flush()
    tb_writer.close()


def _retrain_hybrid(*, model, lstm_retrain_opt, xl_retrain_opt, lstm_lr, xl_lr,
                    file_data, file_pos, split, batch_size_streaming,
                    retrain_block_len, retrain_seq_length,
                    retrain_batch_size_lstm,
                    retrain_tgt_len, retrain_mem_len,
                    stream_mem_len, vocab_size, device):
  """Retrain both submodels on the same trailing window in their native shapes.

  - LSTM: BPTT batches of (retrain_seq_length+1)-token sequences over the
    last ``retrain_block_len`` chars, sharded across the same parallel-stream
    layout the streaming loop uses (i.e. each stream's history is its own
    column). This mirrors the data-construction code embedded in the LSTM
    solo retrain block of ``process()`` exactly; if that body changes,
    update here.
  - Transformer-XL: NNCP-style streaming pass via ``_retrain_transformer_xl``,
    which itself reset_lengths the model to (retrain_tgt_len, 0,
    retrain_mem_len) and back. Each retrain pass on the trailing
    ``retrain_block_len`` chars (capped at retrain_block_len for our
    transformer's retrain_block_len which is independent of the LSTM one).

  Both retrains see the same recent data but in different shapes; each
  updates only its own submodel.
  """
  # ---- LSTM half ----
  pos = file_pos - 1  # streaming step at retrain trigger
  r_start_step = max(0, pos - retrain_block_len)
  r_end_step = pos

  all_inputs = []
  all_targets = []
  r_step = r_start_step
  while r_step < r_end_step:
    for i in range(batch_size_streaming):
      base_idx = r_step + i * split
      start_idx = base_idx
      end_idx = start_idx + retrain_seq_length + 1
      current_stream_limit = i * split + pos + 1
      stream_segment = file_data[start_idx: min(end_idx, current_stream_limit)]
      if len(stream_segment) < retrain_seq_length + 1:
        stream_segment = list(stream_segment) + [0] * (retrain_seq_length + 1 - len(stream_segment))
      all_inputs.append(stream_segment[:-1])
      all_targets.append(stream_segment[1:])
    r_step += retrain_seq_length

  if all_inputs:
    all_inputs_t = torch.tensor(all_inputs, dtype=torch.long, device=device)
    all_targets_t = torch.tensor(all_targets, dtype=torch.long, device=device)

    total_examples = all_inputs_t.shape[0]
    remainder = total_examples % retrain_batch_size_lstm
    if remainder != 0:
      all_inputs_t = all_inputs_t[:-remainder]
      all_targets_t = all_targets_t[:-remainder]
      total_examples -= remainder

    if total_examples > 0:
      for i in range(0, total_examples, retrain_batch_size_lstm):
        batch_inputs = all_inputs_t[i: i + retrain_batch_size_lstm]
        batch_targets = all_targets_t[i: i + retrain_batch_size_lstm]
        # Reuse the same retrain_step helper used by the LSTM solo path,
        # but with a single-element model list and the LSTM submodel.
        retrain_step([model.lstm], [lstm_retrain_opt], batch_inputs, batch_targets, lstm_lr)

  # ---- Transformer-XL half ----
  # Reuse the existing _retrain_transformer_xl helper exactly as the
  # transformer solo path does; pass [model.transformer_xl] as the
  # single-element models list.
  _retrain_transformer_xl(
      models=[model.transformer_xl],
      retrain_optimizers=[xl_retrain_opt],
      current_lr=xl_lr,
      file_data=file_data,
      file_pos=file_pos,
      retrain_block_len=retrain_block_len,
      retrain_tgt_len=retrain_tgt_len,
      retrain_mem_len=retrain_mem_len,
      retrain_batch_size=retrain_batch_size_lstm,
      stream_mem_len=stream_mem_len,
      vocab_size=vocab_size,
      device=device,
  )


In [ ]:
#@title Arithmetic Coding Library

#
# Reference arithmetic coding
# Copyright (c) Project Nayuki
#
# https://www.nayuki.io/page/reference-arithmetic-coding
# https://github.com/nayuki/Reference-arithmetic-coding
#

import sys
python3 = sys.version_info.major >= 3


# ---- Arithmetic coding core classes ----

# Provides the state and behaviors that arithmetic coding encoders and decoders share.
class ArithmeticCoderBase(object):

	# Constructs an arithmetic coder, which initializes the internal code range.
	def __init__(self, numbits):
		if numbits < 1:
			raise ValueError("State size out of range")

		# -- Configuration fields --
		# Number of bits for the 'low' and 'high' state variables. Must be at least 1.
		self.num_state_bits = numbits
		# Maximum range (high+1-low) during coding (trivial), which is 2^num_state_bits = 1000...000.
		self.full_range = 1 << self.num_state_bits
		# The top bit at width num_state_bits, which is 0100...000.
		self.half_range = self.full_range >> 1
		# The second highest bit at width num_state_bits, which is 0010...000. This is zero when num_state_bits=1.
		self.quarter_range = self.half_range >> 1
		# Minimum range (high+1-low) during coding (non-trivial), which is 0010...010.
		self.minimum_range = self.quarter_range + 2
		# Maximum allowed total from a frequency table at all times during coding.
		self.maximum_total = self.minimum_range
		# Bit mask of num_state_bits ones, which is 0111...111.
		self.state_mask = self.full_range - 1

		# -- State fields --
		# Low end of this arithmetic coder's current range.
		self.low = 0
		# High end of this arithmetic coder's current range.
		self.high = self.state_mask
        
	def get_state(self):
		return {
			'low': self.low,
			'high': self.high
		}
	
	def set_state(self, state):
		self.low = state['low']
		self.high = state['high']


	# Updates the code range (low and high) of this arithmetic coder as a result
	# of processing the given symbol with the given frequency table.
	def update(self, freqs, symbol):
		# State check
		low = self.low
		high = self.high
		range = high - low + 1

		# Frequency table values check
		total = int(freqs[-1])
		symlow = int(freqs[symbol-1]) if symbol > 0 else 0
		symhigh = int(freqs[symbol])

		# Update range
		newlow  = low + symlow  * range // total
		newhigh = low + symhigh * range // total - 1
		self.low = newlow
		self.high = newhigh

		# While low and high have the same top bit value, shift them out
		while ((self.low ^ self.high) & self.half_range) == 0:
			self.shift()
			self.low  = ((self.low  << 1) & self.state_mask)
			self.high = ((self.high << 1) & self.state_mask) | 1
		# Now low's top bit must be 0 and high's top bit must be 1

		# While low's top two bits are 01 and high's are 10, delete the second highest bit of both
		while (self.low & ~self.high & self.quarter_range) != 0:
			self.underflow()
			self.low = (self.low << 1) ^ self.half_range
			self.high = ((self.high ^ self.half_range) << 1) | self.half_range | 1


	# Called to handle the situation when the top bit of 'low' and 'high' are equal.
	def shift(self):
		raise NotImplementedError()


	# Called to handle the situation when low=01(...) and high=10(...).
	def underflow(self):
		raise NotImplementedError()


class ArithmeticEncoder(ArithmeticCoderBase):

	# Constructs an arithmetic coding encoder based on the given bit-level output stream.
	def __init__(self, numbits, bitout):
		super(ArithmeticEncoder, self).__init__(numbits)
		# The underlying bit output stream.
		self.output = bitout
		# Number of saved underflow bits. This value can grow without bound.
		self.num_underflow = 0
		
	def get_state(self):
		state = super(ArithmeticEncoder, self).get_state()
		state['num_underflow'] = self.num_underflow
		return state
	
	def set_state(self, state):
		super(ArithmeticEncoder, self).set_state(state)
		self.num_underflow = state['num_underflow']


	# Encodes the given symbol based on the given frequency table.
	# This updates this arithmetic coder's state and may write out some bits.
	def write(self, freqs, symbol):
		self.update(freqs, symbol)


	# Terminates the arithmetic coding by flushing any buffered bits, so that the output can be decoded properly.
	# It is important that this method must be called at the end of the each encoding process.
	def finish(self):
		self.output.write(1)


	def shift(self):
		bit = self.low >> (self.num_state_bits - 1)
		self.output.write(bit)

		# Write out the saved underflow bits
		for _ in range(self.num_underflow):
			self.output.write(bit ^ 1)
		self.num_underflow = 0


	def underflow(self):
		self.num_underflow += 1


# Reads from an arithmetic-coded bit stream and decodes the stream back into symbols.
class ArithmeticDecoder(ArithmeticCoderBase):

	# Constructs an arithmetic coding decoder based on the
	# given bit input stream, and fills the initial code bits.
	def __init__(self, numbits, bitin):
		super(ArithmeticDecoder, self).__init__(numbits)
		# The underlying bit input stream.
		self.input = bitin
		# The current raw code bits being buffered, which is always in the range [low, high].
		self.code = 0
		for _ in range(self.num_state_bits):
			self.code = self.code << 1 | self.read_code_bit()
	
	def get_state(self):
		state = super(ArithmeticDecoder, self).get_state()
		state['code'] = self.code
		return state
	
	def set_state(self, state):
		super(ArithmeticDecoder, self).set_state(state)
		self.code = state['code']


	# Decodes the next symbol based on the given frequency table and returns it.
	# Also updates this arithmetic coder's state and may read in some bits.
	def read(self, freqs):

		# Translate from coding range scale to frequency table scale
		total = int(freqs[-1])
		range = self.high - self.low + 1
		offset = self.code - self.low
		value = ((offset + 1) * total - 1) // range

		# A kind of binary search. Find highest symbol such that freqs[symbol-1] <= value.
		start = 0
		end = len(freqs)
		while end - start > 1:
			middle = (start + end) >> 1
			low = int(freqs[middle-1]) if middle > 0 else 0
			if low > value:
				end = middle
			else:
				start = middle

		symbol = start
		self.update(freqs, symbol)
		return symbol


	def shift(self):
		self.code = ((self.code << 1) & self.state_mask) | self.read_code_bit()


	def underflow(self):
		self.code = (self.code & self.half_range) | ((self.code << 1) & (self.state_mask >> 1)) | self.read_code_bit()


	# Returns the next bit (0 or 1) from the input stream.
	def read_code_bit(self):
		temp = self.input.read()
		if temp == -1:
			temp = 0
		return temp


# ---- Bit-oriented I/O streams ----

# A stream of bits that can be read. The bits are read in big-endian order.
class BitInputStream(object):

	# Constructs a bit input stream based on the given byte input stream.
	def __init__(self, inp):
		# The underlying byte stream to read from
		self.input = inp
		# Either in the range [0x00, 0xFF] if bits are available, or -1 if end of stream is reached
		self.currentbyte = 0
		# Number of remaining bits in the current byte, always between 0 and 7 (inclusive)
		self.numbitsremaining = 0
  
	def get_state(self):
		return {
			'currentbyte': self.currentbyte,
			'numbitsremaining': self.numbitsremaining
		}
	
	def set_state(self, state):
		self.currentbyte = state['currentbyte']
		self.numbitsremaining = state['numbitsremaining']


	# Reads a bit from this stream. Returns 0 or 1 if a bit is available, or -1 if EOF.
	def read(self):
		if self.currentbyte == -1:
			return -1
		if self.numbitsremaining == 0:
			temp = self.input.read(1)
			if len(temp) == 0:
				self.currentbyte = -1
				return -1
			self.currentbyte = temp[0] if python3 else ord(temp)
			self.numbitsremaining = 8
		assert self.numbitsremaining > 0
		self.numbitsremaining -= 1
		return (self.currentbyte >> self.numbitsremaining) & 1


	# Reads a bit from this stream. Returns 0 or 1, or raises an EOFError if EOF.
	def read_no_eof(self):
		result = self.read()
		if result != -1:
			return result
		else:
			raise EOFError()


	# Closes this stream and the underlying input stream.
	def close(self):
		self.input.close()
		self.currentbyte = -1
		self.numbitsremaining = 0


# A stream where bits can be written. Padded with 0's to align on byte boundaries.
class BitOutputStream(object):

	# Constructs a bit output stream based on the given byte output stream.
	def __init__(self, out):
		self.output = out  # The underlying byte stream to write to
		self.currentbyte = 0  # The accumulated bits for the current byte
		self.numbitsfilled = 0  # Number of accumulated bits in the current byte

	def get_state(self):
		return {
			'currentbyte': self.currentbyte,
			'numbitsfilled': self.numbitsfilled
		}
	
	def set_state(self, state):
		self.currentbyte = state['currentbyte']
		self.numbitsfilled = state['numbitsfilled']

	# Writes a single bit to the stream.
	def write(self, b):
		if b not in (0, 1):
			raise ValueError("Argument must be 0 or 1")
		self.currentbyte = (self.currentbyte << 1) | b
		self.numbitsfilled += 1
		if self.numbitsfilled == 8:
			towrite = bytes((self.currentbyte,)) if python3 else chr(self.currentbyte)
			self.output.write(towrite)
			self.currentbyte = 0
			self.numbitsfilled = 0


	# Closes this stream. Writes padding to complete the last byte if partially filled.
	def close(self):
		while self.numbitsfilled != 0:
			self.write(0)
		self.output.close()

## Compress

In [ ]:
#@title Preprocess

if mode != 'decompress':
  input_path = path_to_file

  if preprocess == 'cmix':
    !./cmix/cmix -s ./cmix/dictionary/english.dic $path_to_file ./data/preprocessed.dat
    input_path = "./data/preprocessed.dat"

  # Parse the file characters into the sequential int_list buffer.
  int_list = []
  if preprocess == 'nncp' or preprocess == 'nncp-done':
    if preprocess == 'nncp':
      !time ./nncp/preprocess c data/dictionary.words $path_to_file data/preprocessed.dat $n_words $min_freq
    else:
      !cp $path_to_file data/preprocessed.dat
    input_path = "./data/preprocessed.dat"
    orig = open(input_path, 'rb').read()
    for i in range(0, len(orig), 2):
      int_list.append(orig[i] * 256 + orig[i+1])
    vocab_size = int(subprocess.check_output(
        ['wc', '-l', 'data/dictionary.words']).split()[0])
  else:
    text = open(input_path, 'rb').read()
    vocab = sorted(set(text))
    vocab_size = len(vocab)
    # Create a mapping from unique characters to sequential indexes.
    char2idx = {u:i for i, u in enumerate(vocab)}
    for idx, c in enumerate(text):
      int_list.append(char2idx[c])

  # Round the vocabulary size up to a multiple of 8 to improve processing performance.
  vocab_size = math.ceil(vocab_size/8) * 8
  
  if checkpoint:
      ckpt_dir = 'data/ckpt'
      if not os.path.exists(ckpt_dir):
          os.makedirs(ckpt_dir)
  else:
      ckpt_dir = None

  file_len = len(int_list)
  print('Length of file: {} symbols'.format(file_len))
  print('Vocabulary size: {}'.format(vocab_size))

In [ ]:
#@title Compression

if mode == 'compress' or mode == 'both':
  original_file = path_to_file
  path_to_file = "data/compressed.dat"
  
  # Determine correct file operational mode depending on checkpoint part status
  if current_part > 1:
      open_mode = "ab"
      print(f"Resuming compression (part {current_part}/{total_parts}). Appending to {path_to_file}...")
  else:
      open_mode = "wb"
      print(f"Starting compression (part {current_part}/{total_parts})...")

  with open(path_to_file, open_mode) as out:
      # Initialize BitOutputStream. State is tracked via bitout variables so we won't overwrite bits on resume.
      bitout = BitOutputStream(out)
      
      enc = ArithmeticEncoder(32, bitout)

      # Restore previous ArithmeticCoder state if resuming from part > 1.
      if current_part > 1:
          ckpt_dir = os.path.abspath('data/ckpt')
          ac_state_path = os.path.join(ckpt_dir, 'ac_state.pkl')
          if os.path.exists(ac_state_path):
              with open(ac_state_path, 'rb') as f:
                  ac_state = pickle.load(f)
              enc.set_state(ac_state['coder'])
              bitout.set_state(ac_state['bitstream'])
              print("Restored AC state.")
          else:
              print("Warning: Context AC state not found for resuming! Expected corruption.")

      length = len(int_list)
      
      # Prepare and write the overarching file header data if starting fresh
      if current_part == 1:
          # Write the original decoded file length to the compressed file header.
          out.write(length.to_bytes(5, byteorder='big', signed=False))
          if preprocess != 'nncp' and preprocess != 'nncp-done':
              # Without NNCP, explicitly allocate 256 bits for dynamic vocabulary tracking.
              for i in range(256):
                  if i in char2idx:
                      bitout.write(1)
                  else:
                      bitout.write(0)

      # Trigger main compression sequence
      process(True, length, vocab_size, enc, int_list)
      
      # Finish up the bitstream encoding only if this is the final processing chunk.
      if current_part == total_parts:
          enc.finish()
          bitout.close() # Flush and clear padding
      else:
          # On pause, hold off safely closing the bitout to avoid injecting padding bits.
          print(f"Pausing compression part {current_part}. AC state safely packaged for next execution.")
          pass

  print("Compressed size:", os.path.getsize(path_to_file))

In [ ]:
#@title Download Result

if mode == 'preprocess_only':
  if preprocess == 'nncp':
    download('data/dictionary.words')
  download(input_path)
elif mode != 'decompress':
  download('data/compressed.dat')
  if preprocess == 'nncp':
    download('data/dictionary.words')
  if checkpoint and mode != "both":
    # Bundle the data/ckpt directory (model weights, optimizer state, AC state)
    # into a single zip for download.
    import shutil
    shutil.make_archive('data/ckpt', 'zip', 'data/ckpt')
    download('data/ckpt.zip')

## Decompress

In [ ]:
#@title Decompression

if mode == 'decompress' or mode == 'both':
  output_path = "data/decompressed.dat"
  
  # Check for appropriate resuming status and partial files
  if current_part > 1:
      if not os.path.exists(output_path):
          print(f"Error: Attempting to resume decompression (part {current_part}), but {output_path} could not be found.")
          print("Please ensure the partial decompressed file was uploaded successfully.")
      else:
           print(f"Resuming decompression (part {current_part}/{total_parts}). Targetting {output_path}...")
  else:
      print(f"Starting decompression (part {current_part}/{total_parts})...")

  # Open the binary input path to interpret compressed sequence arrays
  with open(path_to_file, "rb") as inp:
      # Read the expected original file chunk length from the 5-byte header limit.
      length = int.from_bytes(inp.read()[:5], byteorder='big')
      
      # Allocate and prep the resulting symbol character buffer.
      output = [0] * length
      if current_part > 1:
           
           print("Loading partially decompressed context block...")
           if os.path.exists(output_path):
                # Retrieve standard file contents for context continuation matching.
                with open(output_path, "rb") as f_partial:
                    partial_data = f_partial.read()
                    
                # Reconstruct values safely mapping directly onto pre-processor logic.
                if preprocess == 'nncp' or preprocess == 'nncp-done':
                    for i in range(len(partial_data) // 2):
                        val = partial_data[2*i] * 256 + partial_data[2*i+1]
                        output[i] = val
                else:
                    # Decompressor will map reverse indices further downstream
                    pass 

      inp.seek(5)
      bitin = BitInputStream(inp)

      if preprocess == 'nncp' or preprocess == 'nncp-done':
        # Rely on fixed, precompiled dictionary vocab sizes via NNCP
        vocab_size = int(subprocess.check_output(
            ['wc', '-l', 'data/dictionary.words']).split()[0])
      else:
        # Check standard 256 byte matrix blocks to calculate explicit length
        vocab = []
        for i in range(256):
          if bitin.read():
            vocab.append(i)
        vocab_size = len(vocab)
        
        # Determine strict char2idx conversions mapping directly on file updates
        if current_part > 1 and not (preprocess == 'nncp' or preprocess == 'nncp-done'):
             char2val = {c: i for i, c in enumerate(vocab)}
             
             # Safely reverse match the expected character outputs
             for i in range(len(partial_data)):
                 c = partial_data[i]
                 if c in char2val:
                     output[i] = char2val[c]
                 else:
                     pass


      # Round up vocabulary limits tracking to multiples of 8 for pipeline efficiency.
      vocab_size = math.ceil(vocab_size/8) * 8
      
      dec = ArithmeticDecoder(32, bitin)
      
      # Refresh AC configurations from preserved memory checkpoints.
      if current_part > 1:
          ckpt_dir = os.path.abspath('data/ckpt')
          ac_state_path = os.path.join(ckpt_dir, 'ac_state.pkl')
          
          if os.path.exists(ac_state_path):
              with open(ac_state_path, 'rb') as f:
                  ac_state = pickle.load(f)
              
              dec.set_state(ac_state['coder'])
              bitin.set_state(ac_state['bitstream'])
              
              # Re-align raw file buffer offsets specifically targeted by stream positions
              if 'file_pos' in ac_state:
                  inp.seek(ac_state['file_pos'])
              else:
                   print("Warning: Input target stream position unverified! AC pointer lost.")
                   
              print(f"Restored continuous AC checkpoint state. Current buffer offset: {inp.tell()}")

          else:
              print("Warning: Critical AC parameters skipped. Resumption integrity unverified.")

      # Trigger main decompression sequence
      process(False, length, vocab_size, dec, output)
      
      # Set explicit read/write target modes accounting for fragmented file architectures.
      mode_write = "r+b" if (current_part > 1 and os.path.exists(output_path)) else "wb"
      
      with open(output_path, mode_write) as out:
        if preprocess == 'nncp' or preprocess == 'nncp-done':
            for i in range(length):
                out.write(bytes(((output[i] // 256),)))
                out.write(bytes(((output[i] % 256),)))
        else:
            # Map index translations properly tracking original character outputs.
            idx2char = np.array(vocab)
            for i in range(length):
                 out.write(bytes((idx2char[output[i]],)))

  if preprocess == 'cmix':
    if current_part == total_parts:
        !./cmix/cmix -d ./cmix/dictionary/english.dic $output_path ./data/final.dat
        output_path = "data/final.dat"
  if preprocess == 'nncp' or preprocess == 'nncp-done':
    if current_part == total_parts:
        !./nncp/preprocess d data/dictionary.words $output_path ./data/final.dat
        output_path = "data/final.dat"

In [ ]:
#@title Download Result

if mode == 'decompress':
  if preprocess == 'nncp-done':
    download('data/decompressed.dat')
  else:
    download(output_path)
  if checkpoint:
    import shutil
    shutil.make_archive('data/ckpt', 'zip', 'data/ckpt')
    download('data/ckpt.zip')

In [ ]:
#@title Validation

if mode == 'decompress' or mode == 'both':
  if preprocess == 'nncp-done':
    !md5sum data/decompressed.dat
  !md5sum $output_path
if mode == 'both':
  !md5sum $original_file

In [ ]:
#@title Close Runtime

import time
time.sleep(60)

from google.colab import runtime
runtime.unassign()